# zip() 語法詳細解說
Status: in_use  
Prereq: Python 基礎語法、input()/split()、for 迴圈  
Can-do checklist:  
- [ ] 能用 zip 進行並行走訪
- [ ] 能理解 zip 是 iterator（只能走一次）
- [ ] 能處理長度不一致（strict/zip_longest）
- [ ] 能用 zip(*pairs) 做 unzip/轉置並處理空資料邊界
- [ ] 能完成 zip 改錯題（Fix-it）


這份筆記系統整理 Python 內建的 `zip()`：
- 最基本語法與輸出型態（iterator）
- 常見用法：並行走訪、配對、轉置、解包 (unzip)
- 不同長度序列的處理（`zip` vs `itertools.zip_longest`，以及 Python 3.10+ 的 `strict=True`）

本課程建議你把 `zip` 當成「把多個序列對齊後，一次拿出同位置元素」的工具。


## 1) 什麼是 zip()

`zip(a, b, c, ...)` 會回傳一個 iterator，每次產出一個 tuple：

- 第 1 次：`(a[0], b[0], c[0], ...)`
- 第 2 次：`(a[1], b[1], c[1], ...)`
- ...

預設行為：**走到最短的序列就停止**。


In [ ]:
a = [10, 20, 30]
b = ['x', 'y', 'z']

pairs = zip(a, b)
print(pairs)              # zip object (iterator)
print(list(pairs))        # 轉成 list 才會看到內容


## 2) zip 是 iterator：只能走一次

很多新手踩雷：你把 `zip(...)` 先 `list()` 看過一次後，它就「被消耗完了」。


In [ ]:
a = [1, 2, 3]
b = [4, 5, 6]

z = zip(a, b)
print(list(z))
print(list(z))  # 第二次會是空的


### 2-1) iterator 到底是什麼？（Iterable vs Iterator）

在 Python 裡你常會聽到兩個字：

- **Iterable（可迭代物）**：可以被 `for` 走訪的東西（例如 `list`、`tuple`、`str`）。
- **Iterator（迭代器）**：每次只能「往前取下一個」的東西。

重要差異：
- `list` 你可以重複走訪很多次。
- `zip` / `map` / `filter` 產生的是 **iterator**：走過一次就被消耗。


In [ ]:
# iterable: list
nums = [1, 2, 3]
print(iter(nums))         # 你可以把 iterable 變成 iterator

# iterator: zip
z = zip([1, 2, 3], [10, 20, 30])
print(z)


### 2-2) iter() / next() / StopIteration（最核心的 3 個概念）

`for x in something:` 背後其實在做：
1. `it = iter(something)`
2. 不斷 `next(it)` 取下一個
3. 取到沒東西時會丟 `StopIteration`，`for` 會自動停止

你可以自己手動操作一次，看 iterator 怎麼「一路往前走」。


In [ ]:
z = zip([1, 2], ['a', 'b'])
it = iter(z)

print(next(it))
print(next(it))

try:
    print(next(it))
except StopIteration:
    print('StopIteration: 沒有下一個了')


### 2-3) 為什麼你會覺得「沒從頭開始」？

因為 iterator 不是「容器」，它比較像一條管線：資料經過就不見了。

所以如果你要再用一次：
- **重新呼叫** `zip(...)` 產生新的 iterator
- 或者先 `list(...)` 把結果存起來（但會占記憶體）


In [ ]:
a = [1, 2, 3]
b = [4, 5, 6]

z = zip(a, b)
print(list(z))

# 再用一次：必須重建
z = zip(a, b)
print(list(z))


## 3) 最常見：並行走訪（parallel iteration）

當你需要「同時拿到兩個（或多個）陣列的同位置元素」時，用 `zip` 最乾淨。


In [ ]:
names = ["Amy", "Bob", "Cora"]
scores = [90, 75, 100]

for name, score in zip(names, scores):
    print(name, score)


## 4) 組成 dict、或產生配對資料

常見：把兩個列表合成字典（key/value）。


In [ ]:
keys = ["a", "b", "c"]
values = [1, 2, 3]

d = dict(zip(keys, values))
print(d)


## 5) 不同長度時 zip 會截斷

`zip([1,2,3], [10,20])` 只會產出 2 組，因為第二個序列已經結束。

這個行為有時是優點（避免越界），但有時會掩蓋 bug。


In [ ]:
a = [1, 2, 3]
b = [10, 20]

print(list(zip(a, b)))


## 6) Python 3.10+：zip(..., strict=True) 讓長度不一致直接報錯

如果你期待兩個序列一定等長，建議開 `strict=True`，錯了就早點發現。

注意：`strict=True` 只有 Python 3.10 以上才有。


In [ ]:
a = [1, 2, 3]
b = [10, 20]

try:
    print(list(zip(a, b, strict=True)))
except ValueError as e:
    print("ValueError:", e)


## 7) zip_longest：想要補齊時用 itertools

當你希望「走到最長」並用某個預設值補齊短的序列時，使用：

- `itertools.zip_longest(a, b, fillvalue=...)`


In [ ]:
from itertools import zip_longest

a = [1, 2, 3]
b = [10, 20]

print(list(zip_longest(a, b, fillvalue=0)))


## 8) 解包（unzip）：把配對資料拆回兩個序列

如果你有 `pairs = [(a1, b1), (a2, b2), ...]`，想拆回兩個序列：

- 用 `zip(*pairs)`

注意：`pairs` 為空時要小心（會得到空 iterator）。


In [ ]:
pairs = [(1, 'x'), (2, 'y'), (3, 'z')]

left, right = zip(*pairs)
print(left)
print(right)

print(list(left))  # tuple 也可以轉 list


## 9) 轉置（transpose）：把「列」變「欄」

若你把 2D list 想像成矩陣：

- `matrix = [row1, row2, row3, ...]`
- 你想要拿到「每一欄」：`zip(*matrix)`

前提：每一列長度要一致（或你要用 `zip_longest`）。


In [ ]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

cols = list(zip(*matrix))
print(cols)          # 每個元素是一個 tuple（代表一欄）
print(list(cols[0])) # 第一欄


## 10) zip + enumerate：同時拿到索引與多個序列元素

寫法 1：`for i, (a, b) in enumerate(zip(A, B)):`

寫法 2：如果你只需要索引來做輸出排版，也可以不用 enumerate。


In [ ]:
A = [100, 200, 300]
B = [1, 2, 3]

for i, (a, b) in enumerate(zip(A, B)):
    print(i, a + b)


## 11) 常見考題/實務：同時讀兩個陣列做運算

例如：兩串數列相加、計算點積 (dot product)。


In [ ]:
x = [1, 2, 3]
y = [4, 5, 6]

dot = sum(a*b for a, b in zip(x, y))
print(dot)


## 12) 改錯練習（Fix-it）：看得懂 zip 才算真的會

說明：下面每一題都先給一段「有問題的程式」。

你的任務：
1. 先說出它哪裡錯（觀念錯/型態錯/iterator 被消耗/長度不一致被截斷…）
2. 再把它改對

> 這些錯誤在 APCS / ZeroJudge 超常見，因為 `zip` 是 iterator，而且預設會截斷。


### Fix-it 1：zip 走過一次就沒了（iterator 被消耗）


In [ ]:
# 問題：為什麼第二次印出來是空的？請修正。

A = [1, 2, 3]
B = [4, 5, 6]

z = zip(A, B)
print(list(z))
print(list(z))


參考修正方向：
- 方案 A：在第一次就存下結果 `pairs = list(zip(A, B))`
- 方案 B：每次要用都重建 `zip(A, B)`


In [ ]:
# 一種修正範例（方案 A）
A = [1, 2, 3]
B = [4, 5, 6]

pairs = list(zip(A, B))
print(pairs)
print(pairs)


### Fix-it 2：zip 默默截斷，結果少算了


In [ ]:
# 問題：你以為會算到 3 組，其實只算到 2 組。請修正並讓它「明確」處理長度不一致。

A = [10, 20, 30]
B = [1, 2]

print('pairs =', list(zip(A, B)))

# 想做：sum(A[i] * B[i])
dot = sum(a * b for a, b in zip(A, B))
print('dot =', dot)


參考修正方向（擇一）：
- 你期待等長：Python 3.10+ 用 `zip(A, B, strict=True)`，長度不一致就直接報錯。
- 你想補齊：用 `itertools.zip_longest(A, B, fillvalue=0)`。
- 你想手動檢查：先比 `len(A) != len(B)` 就處理。


In [ ]:
# 一種修正範例（strict=True；Python 3.10+）
A = [10, 20, 30]
B = [1, 2]

try:
    dot = sum(a * b for a, b in zip(A, B, strict=True))
    print('dot =', dot)
except ValueError as e:
    print('ValueError:', e)


### Fix-it 3：unzip 空資料會炸（zip(*pairs) 的邊界）


In [2]:
print(  bool( [1] )   )


True


In [ ]:
# 問題：pairs 可能是空的時候，這段會拋 ValueError。請修正。

pairs = []


left, right = zip(*pairs)
print(left, right)


參考修正方向：
- 先判斷空：`if not pairs: ...`
- 或給預設：空時回傳兩個空 tuple / 空 list。


In [ ]:
# 一種修正範例
pairs = []

if not pairs:
    left, right = (), ()
else:
    left, right = zip(*pairs)

print(left, right)


### Fix-it 4：zip(*matrix) 轉置時列長不一樣


In [ ]:
# 問題：你想轉置，但每一列長度不一致，zip 會截斷，結果少欄位。
# 請修正成：要嘛強制等長（抓 bug），要嘛用 zip_longest 補齊。

matrix = [
    [1, 2, 3],
    [4, 5],
    [6, 7, 8],
]

print(list(zip(*matrix)))


In [ ]:
# 一種修正範例（補齊）
from itertools import zip_longest

matrix = [
    [1, 2, 3],
    [4, 5],
    [6, 7, 8],
]

print(list(zip_longest(*matrix, fillvalue=None)))


## 13) 常見錯誤整理

1) `zip` 不是 list：你要印出內容，請 `list(zip(...))` 或用 for 迴圈走訪。

2) `zip` 只能走一次：要重複使用就重新呼叫 `zip(...)`。

3) 長度不一致會被截斷：
   - 不想截斷：Python 3.10+ 用 `strict=True`
   - 或用 `itertools.zip_longest`

4) `zip(*pairs)` 的 `*`：代表把「多個 pair」展開成多個引數。
